# Ноутбук с финальным моделированием

## 1. Импорт бибилиотек и конфигурация проекта

In [35]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
import copy
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_validate
import xgboost as xgb
import pyarrow
import phik
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import catboost
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)
from catboost import CatBoostRegressor
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import plotly
import mlflow
import time
from scipy.stats import randint, uniform, loguniform
import os
from datetime import datetime
import category_encoders as ce
import joblib

In [36]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [37]:
df = pd.read_parquet("../data/raw/df_optimal.parquet")

## 2. Очистка данных

In [38]:
df = df.drop_duplicates()

In [39]:
threshold = len(df) * 0.1
df = df.dropna(thresh=threshold, axis=1).copy()

In [43]:
df = df.drop(columns=['Макро-регион', 'Город', 'Пропуски в данных', 'Ссылка', 'Кол-во просмотров', 'Ошибка_ст', 'Ошибка_знач', 'Скрыто'])

In [41]:
# cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
# for col in cat_cols:
#     df[col] = df[col].fillna('Unknown')

In [44]:
df.isna().sum()

Название машины                    0
Год                                0
Дата размещения объявления         0
Цена                               0
Объем двигателя                   32
Тип двигателя                     21
Мощность                          41
Коробка передач                   22
Привод                            26
Пробег                          4697
Руль                             206
Поколение                         78
Рестайлинг                        78
Цвет                            2213
Комплектация                    1277
Владелец                           0
Тип кузова                      2355
Метка                              0
Регион                             0
Владельцы                     247977
dtype: int64

## 3. Генерация признаков

In [ ]:
def get_brand_tier(brand):
    luxury = {
        "rolls-royce",
        "bentley",
        "lamborghini",
        "ferrari",
        "aston_martin",
        "aston martin",
        "maserati",
        "bugatti",
        "mclaren",
    }

    premium = {
        "bmw",
        "mercedes-benz",
        "audi",
        "lexus",
        "porsche",
        "land_rover",
        "land rover",
        "jaguar",
        "genesis",
        "infiniti",
        "cadillac",
        "volvo",
        "jeep",
        "hongqi",
        "li",
    }

    economy = {"lada", "daewoo", "zaz", "gaz", "uaz", "vaz"}

    if brand in luxury:
        return "Люкс"
    elif brand in premium:
        return "Премиум"
    elif brand in economy:
        return "Эконом"
    else:
        # Сюда попадут все остальные: toyota, nissan, hyundai, kia, volkswagen,
        # chery, geely, haval, peugeot, citroen, subaru, skoda и т.д.
        return "Масс-маркет"


In [ ]:
def create_features(df):
    
    df = df.copy()

    # Создадим другие признаки
    df['Возраст'] = CONFIG["YEAR"] - df['Год']
    df['Пробег за год'] = df['Пробег'] / df['Возраст'].replace(0, 1)  # Чтобы избежать деления на ноль
    df['Литровая мощность'] = (df['Мощность'] / df['Объем двигателя'].replace(0, np.nan)).fillna(0)  # Чтобы избежать деления на ноль

    # Уберем "Название машины", заменив на "Модель"
    df['Название машины'] = df['Название машины'].astype(str)
    df['Марка'] = df['Марка'].astype(str)
    def extract_model(row):
        full_name = row['Название машины']
        # Нормализуем исходную марку (например, "aston_martin" -> "aston martin")
        brand_raw = row['Марка'].lower().replace('_', ' ')
        full_name_lower = full_name.lower()
        
        # 1. Словарь синонимов для брендов, которые могут быть написаны по-русски
        brand_synonyms = {
            'lada': ['лада'],
            'uaz': ['уаз'],
            'gaz': ['газ'],
            'moskvich': ['москвич'],
        }
        
        # 2. Получаем список возможных написаний для текущей марки.
        # Если марки нет в словаре синонимов, используем только саму марку (в списке).
        possible_names = brand_synonyms.get(brand_raw, [brand_raw])
        
        # 3. Проверяем каждый синоним по очереди
        for name in possible_names:
            if full_name_lower.startswith(name):
                # Как только нашли совпадение, отрезаем его длину от оригинального названия
                return full_name[len(name):].strip()
                
        # Если совпадений вообще не нашлось (например, написано просто "Гранта")
        return full_name

    # Применяем новую функцию
    df['Модель'] = df.apply(extract_model, axis=1)

    cat_cols = df.select_dtypes('object').columns.to_list()
    df[cat_cols] = df[cat_cols].astype('category')

    df['Поколение'] = df['Поколение'].astype('category')
    df['Рестайлинг'] = df['Рестайлинг'].astype('category')

    return df